# 🏛️ Train Virgo-1.0-Bayon on a GPU — Colab or Kaggle

The same Virgo training as Virgo-1.0-Angkor (Gemma 3, LoRA in 4 bits), saved as **Virgo-1.0-Bayon**, and Bayon
introduces itself as Bayon. Use this when no TPU is available (`chat/train_tpu_kaggle.ipynb` is the TPU version).

**Size (automatic):** a 24 GB+ GPU (L4, A100) trains the **12B** (~1–2 h). A 16 GB T4 trains the **4B** (~1 h),
so it finishes within free Colab's limits. Set `BASE_MODEL` to force one (12B on a T4 works but takes 3–5 h).

**First time:** on Hugging Face, accept the license for [gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it)
and [gemma-3-12b-it](https://huggingface.co/google/gemma-3-12b-it).

**Run on Colab:** Runtime → Change runtime type → **T4 GPU** (or L4/A100 on Pro); 🔑 Secrets: `HF_TOKEN` (write access),
notebook access on → **Runtime → Run all** and keep the tab open.
**Run on Kaggle:** Accelerator **GPU T4 x2**, Internet on, secret `HF_TOKEN` → **Save & Run All (Commit)**.

It saves **Virgo-1.0-Bayon** (12B) or **Virgo-1.0-Bayon-4B** privately to Hugging Face. The server notebooks then
serve it next to Angkor when there's room (Kaggle T4 x2, or a 24 GB GPU).

In [ ]:
# ⚙️ Settings
BASE_MODEL = "auto"   # "auto" (12B on a 24 GB+ GPU, else 4B), or "google/gemma-3-12b-it" / "google/gemma-3-4b-it"
EPOCHS = 3

In [ ]:
# 1️⃣ Set up
import os, subprocess, sys, glob, json, torch
assert torch.cuda.is_available(), "❌ No GPU: Colab → Runtime → Change runtime type → T4 GPU (Kaggle: Accelerator → GPU)"
KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if KAGGLE else "/content"
memory = torch.cuda.get_device_properties(0).total_memory
print("✅ GPU:", torch.cuda.get_device_name(0), f"({memory / 1e9:.0f} GB)")
if BASE_MODEL == "auto":
    BASE_MODEL = "google/gemma-3-12b-it" if memory > 22e9 else "google/gemma-3-4b-it"
MODEL_NAME = "Virgo-1.0-Bayon-12B" if "12b" in BASE_MODEL else "Virgo-1.0-Bayon-4B"  # Virgo-1.0-Bayon is the 27B (train_cloud.ipynb / train_local.sh)
OUT = "chat/out/virgo-1.0-bayon"
print("Training", BASE_MODEL, "→", MODEL_NAME)

os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"], check=True)
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], capture_output=True, text=True)
if r.returncode:
    print(r.stderr[-3000:]); raise SystemExit("❌ Install failed: send the lines above to Claude.")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # conflicts with newer peft

def secret(name):
    if KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    from getpass import getpass
    return os.environ.get(name) or getpass(f"{name}: ").strip()

from huggingface_hub import login, HfApi, hf_hub_download
login(token=secret("HF_TOKEN"))
ME = HfApi().whoami()["name"]
REPO = f"{ME}/{MODEL_NAME}"
try:  # extra examples made by chat/distill_vertex.ipynb (Gemma 3 27B)
    hf_hub_download(f"{ME}/Virgo-1.0-Angkor-Data", "distilled_gemma27b.jsonl", repo_type="dataset", local_dir="chat/data")
except Exception:
    pass

# Bayon's copy of the data: the same examples, but Virgo calls itself Virgo-1.0-Bayon.
os.makedirs("chat/data_bayon", exist_ok=True)
for path in glob.glob("chat/data/*.jsonl"):
    text = open(path, encoding="utf-8").read().replace("Virgo-1.0-Angkor", "Virgo-1.0-Bayon")
    open(os.path.join("chat/data_bayon", os.path.basename(path)), "w", encoding="utf-8").write(text)
print("✅ Ready:", len(glob.glob("chat/data_bayon/*.jsonl")), "data files → will save to", REPO)

def run(cmd):
    """Runs a step and shows its output; on failure shows the last lines (send them to Claude)."""
    import collections
    tail = collections.deque(maxlen=60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end=""); tail.append(line)
    if proc.wait() != 0:
        print("\n" + "=" * 60 + "\n❌ It stopped with an error. Send these last lines to Claude:\n" + "=" * 60)
        print("".join(tail))
        raise SystemExit(f"❌ {next((a for a in cmd if a.endswith('.py')), cmd[0])} failed")

In [ ]:
# 2️⃣ Train Virgo-1.0-Bayon
run([sys.executable, "scripts/check_data.py", "chat/data_bayon/*.jsonl"])
run([sys.executable, "-u", "chat/train.py", "--base", BASE_MODEL, "--data", "chat/data_bayon/*.jsonl", "--epochs", str(EPOCHS), "--out", OUT])

In [ ]:
# 3️⃣ Try Bayon
sys.path.append("chat")
from virgo_chat import VirgoChat
bot = VirgoChat(adapter=OUT)
for q in ["Who are you?", "អ្នកជានរណា?", "What is 15% of 200?", "ពន្យល់ AI ឱ្យងាយយល់", "Translate to Khmer: Thank you for your help."]:
    print("🧑", q); print("🤖", bot.reply([{"role": "user", "content": q}], temperature=0), "\n")

In [ ]:
# 4️⃣ Save to Hugging Face (private)
open(f"{OUT}/README.md", "w", encoding="utf-8").write(f"""---
base_model: {BASE_MODEL}
library_name: peft
license: gemma
language: [en, km]
tags: [virgo, ksn, khmer, lora]
---
# {MODEL_NAME}

KSN's {MODEL_NAME} for [Virgo AI](https://virgoai.camksn.com): a LoRA fine-tune of Google's [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL})
on Virgo's own conversations. Use is subject to the [Gemma Terms of Use](https://ai.google.dev/gemma/terms).
""")
api = HfApi()
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO, commit_message=MODEL_NAME, ignore_patterns=["checkpoint-*"])
print(f"✅ Saved {MODEL_NAME}: https://huggingface.co/{REPO}")
print("Next: restart a server notebook with room for two models (Kaggle T4 x2, or a 24 GB GPU) — Bayon appears on the website.")